In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# RealMLP + TabArena — Piloto do Modelo do Grupo no Kaggle

Este notebook executa um piloto mínimo do modelo do grupo/RealMLP usando o runner `src/pipeline/run_all.py`.

Objetivos:
- reutilizar o pacote Kaggle já anexado como Dataset;
- copiar o projeto para `/kaggle/working/project`;
- instalar o projeto local;
- validar imports;
- executar apenas um `task_id`;
- incluir o modelo do grupo com `--include-group-model`;
- salvar métricas de treino e teste em `/kaggle/working/results`.

Restrições:
- não executar os 30 datasets;
- não executar AutoGluon Default;
- não executar AutoGluon Extreme;
- não executar HPO;
- não alterar a lista oficial `RECOMMENDED_TASK_IDS`;
- não salvar resultados finais fora de `/kaggle/working/results`.

Este notebook é um piloto operacional para confirmar se o RealMLP/modelo do grupo roda no Kaggle.

In [1]:
# Esta célula lista recursivamente o conteúdo de /kaggle/input.
# Objetivo:
# - confirmar como o Dataset foi montado pelo Kaggle;
# - identificar o caminho real do projeto anexado;
# - verificar se o projeto já veio descompactado.

from pathlib import Path

input_root = Path("/kaggle/input")

print("Existe /kaggle/input?", input_root.exists())
print("\nConteúdo recursivo de /kaggle/input:")

all_paths = sorted(input_root.rglob("*"))

if not all_paths:
    print("Nenhum arquivo encontrado dentro de /kaggle/input.")
else:
    for path in all_paths:
        kind = "DIR " if path.is_dir() else "FILE"
        size = path.stat().st_size if path.is_file() else "-"
        print(f"{kind} | {path} | size={size}")

Existe /kaggle/input? True

Conteúdo recursivo de /kaggle/input:
DIR  | /kaggle/input/datasets | size=-
DIR  | /kaggle/input/datasets/kischenah | size=-
DIR  | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package | size=-
FILE | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/README.md | size=6910
DIR  | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/data | size=-
FILE | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/data/__init__.py | size=0
FILE | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/data/load_tabarena.py | size=6337
DIR  | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/docs | size=-
FILE | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/docs/HISTORY.md | size=17898
FILE | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/docs/INFRAESTRUTURA.md | size=8840
FILE | /kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package/docs/KAGGLE_CHEC

In [2]:
# Esta célula define o caminho do projeto dentro de /kaggle/input.
# Como o Kaggle já montou o pacote descompactado, não precisamos procurar nem extrair ZIP.

from pathlib import Path

input_project_dir = Path("/kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package")

assert input_project_dir.exists(), f"Projeto não encontrado em: {input_project_dir}"
assert (input_project_dir / "src").exists(), "Diretório src não encontrado."
assert (input_project_dir / "data").exists(), "Diretório data não encontrado."
assert (input_project_dir / "notebooks").exists(), "Diretório notebooks não encontrado."
assert (input_project_dir / "pyproject.toml").exists(), "pyproject.toml não encontrado."

print("Projeto encontrado em:")
print(input_project_dir)

Projeto encontrado em:
/kaggle/input/datasets/kischenah/realmlp-tabarena-kaggle-package


In [3]:
# Esta célula copia o projeto de /kaggle/input para /kaggle/working/project.
#
# Motivo:
# - /kaggle/input é somente leitura;
# - /kaggle/working é gravável;
# - os scripts podem precisar criar arquivos temporários;
# - os resultados devem ser preservados em /kaggle/working/results.

import shutil
from pathlib import Path

project_dir = Path("/kaggle/working/project")
results_dir = Path("/kaggle/working/results")

if project_dir.exists():
    shutil.rmtree(project_dir)

shutil.copytree(input_project_dir, project_dir)

results_dir.mkdir(parents=True, exist_ok=True)

print("Projeto copiado para:")
print(project_dir)

print("\nDiretório de resultados criado/confirmado:")
print(results_dir)

Projeto copiado para:
/kaggle/working/project

Diretório de resultados criado/confirmado:
/kaggle/working/results


In [4]:
# Esta célula confirma a estrutura principal do projeto já copiado para /kaggle/working/project.
# Ela serve como verificação antes de instalar dependências ou executar scripts.

from pathlib import Path

print("Conteúdo de /kaggle/working/project:")

for path in sorted(project_dir.iterdir()):
    print("-", path.name)

required_paths = [
    project_dir / "src",
    project_dir / "data",
    project_dir / "notebooks",
    project_dir / "docs",
    project_dir / "pyproject.toml",
]

for path in required_paths:
    assert path.exists(), f"Item obrigatório ausente: {path}"

print("\nEstrutura mínima validada.")

Conteúdo de /kaggle/working/project:
- README.md
- data
- docs
- notebooks
- pyproject.toml
- src

Estrutura mínima validada.


In [5]:
# Esta célula define /kaggle/working/project como diretório atual.
# A partir daqui, os comandos são executados dentro da raiz do projeto.

import os
from pathlib import Path

os.chdir(project_dir)

print("Diretório atual:")
print(Path.cwd())

Diretório atual:
/kaggle/working/project


In [6]:
# Esta célula verifica quais arquivos de dependência existem no pacote.
# No pacote atual foi identificado pyproject.toml, então a instalação deve usar o projeto local.

from pathlib import Path

dependency_files = [
    "requirements.txt",
    "requirements-dev.txt",
    "pyproject.toml",
]

for filename in dependency_files:
    path = Path(filename)
    print(filename, "=>", "existe" if path.exists() else "não existe")

requirements.txt => não existe
requirements-dev.txt => não existe
pyproject.toml => existe


In [7]:
# Esta célula instala o projeto local em modo editável.
# Objetivo:
# - permitir imports de src, data e módulos internos;
# - instalar dependências declaradas no pyproject.toml;
# - preparar o ambiente para um piloto pequeno.

import sys
import subprocess

cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-e",
    ".",
]

print("Executando:")
print(" ".join(cmd))

subprocess.check_call(cmd)

print("\nInstalação concluída.")

Executando:
/usr/bin/python3 -m pip install -e .
Obtaining file:///kaggle/working/project
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.5 requires pyarrow>=21.0.0, but you have pyarrow 20.0.0 which is incompatible.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
jupyterlab-lsp 3.10.2 requires jupyterlab<4.0.0a0,>=3.1.0, but you have jupyterlab 4.5.8 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25


Instalação concluída.


In [8]:
# Esta célula valida os imports principais do projeto.
# Também confirma que a lista RECOMMENDED_TASK_IDS mantém os 30 datasets definidos no projeto.

from data.load_tabarena import RECOMMENDED_TASK_IDS

print("Total de task_ids recomendados:", len(RECOMMENDED_TASK_IDS))
print("Task IDs recomendados:")
print(RECOMMENDED_TASK_IDS)

assert len(RECOMMENDED_TASK_IDS) == 30, "A lista RECOMMENDED_TASK_IDS não tem 30 datasets."

print("\nImports e lista de datasets validados.")

Total de task_ids recomendados: 30
Task IDs recomendados:
[363621, 363629, 363614, 363626, 363685, 363696, 363707, 363671, 363711, 363682, 363684, 363674, 363700, 363702, 363620, 363677, 363704, 363623, 363694, 363706, 363619, 363676, 363712, 363632, 363691, 363681, 363679, 363627, 363613, 363699]

Imports e lista de datasets validados.


In [9]:
# Esta célula fixa o mesmo task_id usado no piloto anterior.
#
# Motivo:
# - permite comparar o comportamento dos baselines e do modelo do grupo;
# - evita executar acidentalmente vários datasets;
# - mantém a execução pequena e rastreável.

pilot_task_id = 363621

print("Task ID piloto selecionado:", pilot_task_id)

Task ID piloto selecionado: 363621


In [10]:
# Esta célula confirma os parâmetros aceitos pelo runner run_all.py.
# O parâmetro crítico para este piloto é --include-group-model.

import subprocess
import sys

script = "src/pipeline/run_all.py"

result = subprocess.run(
    [sys.executable, script, "--help"],
    text=True,
    capture_output=True,
)

print("STDOUT:")
print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

STDOUT:
usage: run_all.py [-h] [--seed SEED] [--train-output TRAIN_OUTPUT]
                  [--test-output TEST_OUTPUT] [--output OUTPUT]
                  [--task-ids [TASK_IDS ...]] [--include-group-model]
                  [--include-hpo] [--hpo-time-limit HPO_TIME_LIMIT]

options:
  -h, --help            show this help message and exit
  --seed SEED
  --train-output TRAIN_OUTPUT
                        caminho do CSV de saída com métricas no conjunto de
                        treinamento
  --test-output TEST_OUTPUT
                        caminho do CSV de saída com métricas no conjunto de
                        teste
  --output OUTPUT       compatibilidade legada: se informado, também salva as
                        métricas de teste neste caminho
  --task-ids [TASK_IDS ...]
                        opcional: lista de task IDs do OpenML; se omitido, usa
                        RECOMMENDED_TASK_IDS
  --include-group-model
                        se passado, inclui o modelo do gr

In [11]:
# Esta célula executa um piloto mínimo com apenas um dataset,
# incluindo o modelo do grupo/RealMLP.
#
# O comando usa:
# - --task-ids 363621: apenas um dataset small;
# - --include-group-model: inclui o modelo da equipe;
# - train-output/test-output separados;
# - saída em /kaggle/working/results.
#
# Não usamos:
# - --include-hpo;
# - AutoGluon;
# - loop nos 30 datasets.

import subprocess
import sys
from pathlib import Path

results_dir = Path("/kaggle/working/results")
results_dir.mkdir(parents=True, exist_ok=True)

pilot_realmlp_train_output = results_dir / f"pilot_realmlp_train_task_{pilot_task_id}.csv"
pilot_realmlp_test_output = results_dir / f"pilot_realmlp_test_task_{pilot_task_id}.csv"
pilot_realmlp_legacy_output = results_dir / f"pilot_realmlp_legacy_test_task_{pilot_task_id}.csv"
pilot_realmlp_log_output = results_dir / f"pilot_realmlp_task_{pilot_task_id}.log"

cmd = [
    sys.executable,
    "src/pipeline/run_all.py",
    "--seed",
    "42",
    "--task-ids",
    str(pilot_task_id),
    "--include-group-model",
    "--train-output",
    str(pilot_realmlp_train_output),
    "--test-output",
    str(pilot_realmlp_test_output),
    "--output",
    str(pilot_realmlp_legacy_output),
]

print("Comando executado:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    text=True,
    capture_output=True,
)

log_text = (
    "COMANDO:\n"
    + " ".join(cmd)
    + "\n\nSTDOUT:\n"
    + result.stdout
    + "\n\nSTDERR:\n"
    + result.stderr
    + f"\n\nRETURN_CODE: {result.returncode}\n"
)

pilot_realmlp_log_output.write_text(log_text, encoding="utf-8")

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

print("Return code:", result.returncode)
print("Log salvo em:", pilot_realmlp_log_output)

assert result.returncode == 0, "Execução piloto com modelo do grupo falhou. Verifique STDOUT/STDERR."

Comando executado:
/usr/bin/python3 src/pipeline/run_all.py --seed 42 --task-ids 363621 --include-group-model --train-output /kaggle/working/results/pilot_realmlp_train_task_363621.csv --test-output /kaggle/working/results/pilot_realmlp_test_task_363621.csv --output /kaggle/working/results/pilot_realmlp_legacy_test_task_363621.csv
STDOUT:

[1/1  100%] blood-transfusion-service-center  n=748  cls=2  reg=small
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best

In [12]:
# Esta célula lista os arquivos gerados em /kaggle/working/results.
# Ela confirma se os artefatos do piloto RealMLP foram salvos corretamente.

from pathlib import Path

results_dir = Path("/kaggle/working/results")

print("Arquivos em /kaggle/working/results:")

for path in sorted(results_dir.rglob("*")):
    if path.is_file():
        print("-", path, "|", path.stat().st_size, "bytes")

Arquivos em /kaggle/working/results:
- /kaggle/working/results/pilot_realmlp_legacy_test_task_363621.csv | 834 bytes
- /kaggle/working/results/pilot_realmlp_task_363621.log | 32754 bytes
- /kaggle/working/results/pilot_realmlp_test_task_363621.csv | 834 bytes
- /kaggle/working/results/pilot_realmlp_train_task_363621.csv | 829 bytes


In [13]:
# Esta célula carrega os CSVs de treino e teste do piloto com modelo do grupo.
# Objetivo:
# - verificar se os arquivos não estão vazios;
# - conferir quais modelos aparecem;
# - confirmar se o modelo do grupo/RealMLP entrou na execução.

import pandas as pd

realmlp_train_df = pd.read_csv(pilot_realmlp_train_output)
realmlp_test_df = pd.read_csv(pilot_realmlp_test_output)

print("TRAIN shape:", realmlp_train_df.shape)
display(realmlp_train_df)

print("\nTEST shape:", realmlp_test_df.shape)
display(realmlp_test_df)

print("\nColunas treino:")
print(list(realmlp_train_df.columns))

print("\nColunas teste:")
print(list(realmlp_test_df.columns))

if "task_id" in realmlp_train_df.columns:
    print("\nTask IDs no treino:", sorted(realmlp_train_df["task_id"].unique()))

if "task_id" in realmlp_test_df.columns:
    print("Task IDs no teste:", sorted(realmlp_test_df["task_id"].unique()))

if "model" in realmlp_train_df.columns:
    print("\nModelos no treino:", sorted(realmlp_train_df["model"].astype(str).unique()))

if "model" in realmlp_test_df.columns:
    print("Modelos no teste:", sorted(realmlp_test_df["model"].astype(str).unique()))

TRAIN shape: (4, 10)


,task_id,dataset,model,auc_ovo,accuracy,g_mean,cross_entropy,fit_time_s,predict_time_s,total_time_s
0,363621,blood-transfusion-service-center,lightgbm,0.774992,0.791587,0.451549,0.464807,3.622441,0.013903,3.636344
1,363621,blood-transfusion-service-center,xgboost,0.881680,0.847036,0.651590,0.366815,0.237875,0.014101,0.251976
2,363621,blood-transfusion-service-center,catboost,0.781702,0.820268,0.618881,0.657172,1.973175,0.014979,1.988154
3,363621,blood-transfusion-service-center,group_model,0.753497,0.797323,0.499009,0.471653,3.333278,0.141903,3.475180



TEST shape: (4, 10)


,task_id,dataset,model,auc_ovo,accuracy,g_mean,cross_entropy,fit_time_s,predict_time_s,total_time_s
0,363621,blood-transfusion-service-center,lightgbm,0.753790,0.768889,0.378087,0.484037,3.622441,0.009010,3.631451
1,363621,blood-transfusion-service-center,xgboost,0.737817,0.768889,0.474610,0.493850,0.237875,0.012524,0.250398
2,363621,blood-transfusion-service-center,catboost,0.724551,0.786667,0.542737,0.661234,1.973175,0.010964,1.984139
3,363621,blood-transfusion-service-center,group_model,0.789690,0.782222,0.461655,0.461845,3.333278,0.139149,3.472427



Colunas treino:
['task_id', 'dataset', 'model', 'auc_ovo', 'accuracy', 'g_mean', 'cross_entropy', 'fit_time_s', 'predict_time_s', 'total_time_s']

Colunas teste:
['task_id', 'dataset', 'model', 'auc_ovo', 'accuracy', 'g_mean', 'cross_entropy', 'fit_time_s', 'predict_time_s', 'total_time_s']

Task IDs no treino: [np.int64(363621)]
Task IDs no teste: [np.int64(363621)]

Modelos no treino: ['catboost', 'group_model', 'lightgbm', 'xgboost']
Modelos no teste: ['catboost', 'group_model', 'lightgbm', 'xgboost']


In [14]:
# Esta célula valida se o piloto RealMLP respeitou as restrições.
#
# Verifica:
# - CSVs não vazios;
# - somente um task_id;
# - task_id correto;
# - AutoGluon não apareceu;
# - Extreme não apareceu;
# - há mais modelos do que no piloto sem RealMLP, idealmente 4 ou mais.

assert len(realmlp_train_df) > 0, "CSV de treino está vazio."
assert len(realmlp_test_df) > 0, "CSV de teste está vazio."

assert "task_id" in realmlp_train_df.columns, "CSV de treino não tem coluna task_id."
assert "task_id" in realmlp_test_df.columns, "CSV de teste não tem coluna task_id."
assert "model" in realmlp_train_df.columns, "CSV de treino não tem coluna model."
assert "model" in realmlp_test_df.columns, "CSV de teste não tem coluna model."

train_task_ids = set(realmlp_train_df["task_id"].astype(int).unique())
test_task_ids = set(realmlp_test_df["task_id"].astype(int).unique())

assert train_task_ids == {int(pilot_task_id)}, f"Treino executou task_ids inesperados: {train_task_ids}"
assert test_task_ids == {int(pilot_task_id)}, f"Teste executou task_ids inesperados: {test_task_ids}"

train_models = set(realmlp_train_df["model"].astype(str).str.lower().unique())
test_models = set(realmlp_test_df["model"].astype(str).str.lower().unique())

combined_models_text = " ".join(sorted(train_models | test_models)).lower()

assert "autogluon" not in combined_models_text, "AutoGluon apareceu nos resultados, o que não era esperado."
assert "extreme" not in combined_models_text, "AutoGluon Extreme apareceu nos resultados, o que não era permitido."

assert len(test_models) >= 4, (
    "Esperava pelo menos 4 modelos no teste: lightgbm, xgboost, catboost e modelo do grupo/RealMLP. "
    f"Modelos encontrados: {sorted(test_models)}"
)

print("Validações do piloto RealMLP concluídas com sucesso.")
print("Task ID executado:", pilot_task_id)
print("Modelos no treino:", sorted(train_models))
print("Modelos no teste:", sorted(test_models))
print("Linhas treino:", len(realmlp_train_df))
print("Linhas teste:", len(realmlp_test_df))

Validações do piloto RealMLP concluídas com sucesso.
Task ID executado: 363621
Modelos no treino: ['catboost', 'group_model', 'lightgbm', 'xgboost']
Modelos no teste: ['catboost', 'group_model', 'lightgbm', 'xgboost']
Linhas treino: 4
Linhas teste: 4


In [15]:
# Esta célula ordena os resultados de teste por AUC, se a coluna existir.
# Objetivo:
# - fazer uma inspeção rápida do desempenho do modelo do grupo no dataset piloto;
# - sem tirar conclusões estatísticas ainda.

metric_candidates = ["auc_ovo", "auc", "accuracy", "acc", "g_mean", "cross_entropy", "time"]

print("Colunas disponíveis:")
print(list(realmlp_test_df.columns))

sort_col = None

for candidate in ["auc_ovo", "auc", "acc", "accuracy"]:
    if candidate in realmlp_test_df.columns:
        sort_col = candidate
        break

if sort_col:
    print(f"\nResultados de teste ordenados por {sort_col}:")
    display(realmlp_test_df.sort_values(sort_col, ascending=False))
else:
    print("\nNenhuma coluna padrão de AUC/ACC encontrada para ordenação automática.")
    display(realmlp_test_df)

Colunas disponíveis:
['task_id', 'dataset', 'model', 'auc_ovo', 'accuracy', 'g_mean', 'cross_entropy', 'fit_time_s', 'predict_time_s', 'total_time_s']

Resultados de teste ordenados por auc_ovo:


,task_id,dataset,model,auc_ovo,accuracy,g_mean,cross_entropy,fit_time_s,predict_time_s,total_time_s
3,363621,blood-transfusion-service-center,group_model,0.789690,0.782222,0.461655,0.461845,3.333278,0.139149,3.472427
0,363621,blood-transfusion-service-center,lightgbm,0.753790,0.768889,0.378087,0.484037,3.622441,0.009010,3.631451
1,363621,blood-transfusion-service-center,xgboost,0.737817,0.768889,0.474610,0.493850,0.237875,0.012524,0.250398
2,363621,blood-transfusion-service-center,catboost,0.724551,0.786667,0.542737,0.661234,1.973175,0.010964,1.984139


In [16]:
# Esta célula cria um manifesto JSON da execução piloto com modelo do grupo.
# O manifesto registra explicitamente que a execução foi fragmentada e não incluiu AutoGluon/HPO.

from pathlib import Path
from datetime import datetime, timezone
import json

manifest = {
    "project": "RealMLP + TabArena",
    "execution_type": "pilot_realmlp_group_model",
    "runner": "src/pipeline/run_all.py",
    "seed": 42,
    "task_id": int(pilot_task_id),
    "train_output": str(pilot_realmlp_train_output),
    "test_output": str(pilot_realmlp_test_output),
    "legacy_output": str(pilot_realmlp_legacy_output),
    "log_output": str(pilot_realmlp_log_output),
    "results_dir": "/kaggle/working/results",
    "ran_all_datasets": False,
    "ran_autogluon_default": False,
    "ran_autogluon_extreme": False,
    "included_group_model": True,
    "included_hpo": False,
    "models_train": sorted(list(train_models)),
    "models_test": sorted(list(test_models)),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

manifest_path = results_dir / f"pilot_realmlp_manifest_task_{pilot_task_id}.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("Manifesto salvo em:")
print(manifest_path)

Manifesto salvo em:
/kaggle/working/results/pilot_realmlp_manifest_task_363621.json


In [17]:
# Esta célula compacta os resultados do piloto RealMLP para download.
# O arquivo ZIP fica em /kaggle/working.

import shutil
from pathlib import Path

archive_base = Path("/kaggle/working/realmlp_tabarena_pilot_realmlp_results")

archive_path = shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=str(results_dir),
)

print("Arquivo ZIP criado:")
print(archive_path)

Arquivo ZIP criado:
/kaggle/working/realmlp_tabarena_pilot_realmlp_results.zip


In [18]:
# Esta célula cria um manifesto JSON da execução piloto com o modelo do grupo.
#
# Observação importante:
# - No CSV bruto, o modelo aparece como "group_model";
# - No projeto, esse identificador corresponde ao RealMLP/modelo da equipe,
#   incluído pelo parâmetro --include-group-model;
# - O manifesto preserva o nome técnico e adiciona um alias legível.

from pathlib import Path
from datetime import datetime, timezone
import json

results_dir = Path("/kaggle/working/results")

model_aliases = {
    "group_model": "RealMLP",
    "lightgbm": "LightGBM",
    "xgboost": "XGBoost",
    "catboost": "CatBoost",
}

manifest = {
    "project": "RealMLP + TabArena",
    "execution_type": "pilot_realmlp_group_model",
    "runner": "src/pipeline/run_all.py",
    "seed": 42,
    "task_id": int(pilot_task_id),

    "train_output": str(pilot_realmlp_train_output),
    "test_output": str(pilot_realmlp_test_output),
    "legacy_output": str(pilot_realmlp_legacy_output),
    "log_output": str(pilot_realmlp_log_output),

    "results_dir": "/kaggle/working/results",

    "ran_all_datasets": False,
    "ran_autogluon_default": False,
    "ran_autogluon_extreme": False,
    "included_group_model": True,
    "included_hpo": False,

    "raw_model_identifier_for_team_model": "group_model",
    "interpreted_model_name_for_team_model": "RealMLP",
    "model_aliases": model_aliases,

    "models_train_raw": sorted(list(train_models)),
    "models_test_raw": sorted(list(test_models)),
    "models_train_display": sorted([model_aliases.get(m, m) for m in train_models]),
    "models_test_display": sorted([model_aliases.get(m, m) for m in test_models]),

    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

manifest_path = results_dir / f"pilot_realmlp_manifest_task_{pilot_task_id}.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("Manifesto salvo em:")
print(manifest_path)

print("\nMapeamento de nomes:")
for raw_name, display_name in model_aliases.items():
    print(f"{raw_name} => {display_name}")

Manifesto salvo em:
/kaggle/working/results/pilot_realmlp_manifest_task_363621.json

Mapeamento de nomes:
group_model => RealMLP
lightgbm => LightGBM
xgboost => XGBoost
catboost => CatBoost


In [20]:
# Esta célula compacta os resultados do piloto RealMLP para download.
# O arquivo ZIP fica em /kaggle/working.

import shutil
from pathlib import Path

archive_base = Path("/kaggle/working/realmlp_tabarena_pilot_realmlp_results")

archive_path = shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=str(results_dir),
)

print("Arquivo ZIP criado:")
print(archive_path)

Arquivo ZIP criado:
/kaggle/working/realmlp_tabarena_pilot_realmlp_results.zip
